In [ ]:
# Wumpus World interactive demo (ipywidgets buttons only)
# Update: when World view is toggled off, it DISAPPEARS and takes NO space.

import random
from dataclasses import dataclass, field
from typing import Tuple, Set, List, Optional, Dict
import ipywidgets as W
from IPython.display import display, HTML

# ---------- Helpers ----------
DIRS = ["E", "S", "W", "N"]  # clockwise
DIR_VEC = {"E": (1, 0), "S": (0, -1), "W": (-1, 0), "N": (0, 1)}
DIR_ARROW = {"E": "→", "S": "↓", "W": "←", "N": "↑"}

def in_bounds(x, y, n=4):
    return 1 <= x <= n and 1 <= y <= n

def neighbors4(x, y, n=4):
    for dx, dy in [(1,0),(-1,0),(0,1),(0,-1)]:
        nx, ny = x+dx, y+dy
        if in_bounds(nx, ny, n):
            yield nx, ny

def percept_list(percepts: Dict[str,bool]) -> List[Optional[str]]:
    order = ["Stench","Breeze","Glitter","Bump","Scream"]
    return [k if percepts.get(k, False) else None for k in order]

# ---------- Core World + Knowledge ----------
@dataclass
class WumpusWorld:
    n: int = 4
    pit_prob: float = 0.2
    rng: random.Random = field(default_factory=random.Random)

    pits: Set[Tuple[int,int]] = field(default_factory=set)
    wumpus: Tuple[int,int] = (2, 2)
    wumpus_alive: bool = True
    gold: Tuple[int,int] = (3, 3)

    ax: int = 1
    ay: int = 1
    facing: str = "E"
    has_gold: bool = False
    arrow_left: int = 1

    done: bool = False
    dead: bool = False
    dead_at: Optional[Tuple[int,int]] = None
    escaped: bool = False
    score: int = 0
    steps: int = 0

    last_bump: bool = False
    last_scream: bool = False

    visited: Set[Tuple[int,int]] = field(default_factory=lambda: {(1,1)})
    log: List[str] = field(default_factory=list)

    kb: Dict[Tuple[int,int], Dict[str,bool]] = field(default_factory=dict)

    def _init_kb(self):
        self.kb = {}
        for x in range(1, self.n+1):
            for y in range(1, self.n+1):
                self.kb[(x,y)] = {
                    "visited": False,
                    "no_pit": False,
                    "poss_pit": False,
                    "no_wumpus": False,
                    "poss_wumpus": False,

                    "known_gold": False,
                    "gold_taken": False,
                    "known_pit": False,
                    "known_wumpus": False,
                    "wumpus_dead_here": False,
                }
        self.kb[(1,1)].update({"visited": True, "no_pit": True, "no_wumpus": True})

    def reset(self, seed: Optional[int] = None):
        if seed is None:
            seed = random.randrange(10**9)
        self.rng = random.Random(seed)

        self.pits = set()
        candidates = [(x,y) for x in range(1,self.n+1) for y in range(1,self.n+1) if (x,y)!=(1,1)]
        self.wumpus = self.rng.choice(candidates)
        self.gold = self.rng.choice(candidates)

        for (x,y) in candidates:
            if self.rng.random() < self.pit_prob:
                self.pits.add((x,y))

        self.wumpus_alive = True
        self.ax, self.ay = 1, 1
        self.facing = "E"
        self.has_gold = False
        self.arrow_left = 1

        self.done = False
        self.dead = False
        self.dead_at = None
        self.escaped = False
        self.score = 0
        self.steps = 0
        self.last_bump = False
        self.last_scream = False

        self.visited = {(1,1)}
        self._init_kb()

        self.log = [f"Reset with seed={seed}. Agent starts at (1,1) facing E."]

        p = self._percepts()
        self._update_kb_from_percepts(p)
        self.log.append(f"Percepts: {percept_list(p)}")
        return seed

    def _percepts(self) -> Dict[str,bool]:
        stench = False
        breeze = False
        glitter = False

        if self.wumpus_alive:
            if (self.ax, self.ay) == self.wumpus:
                stench = True
            else:
                stench = (self.ax, self.ay) in set(neighbors4(*self.wumpus, n=self.n))

        for (px, py) in self.pits:
            if (self.ax, self.ay) in set(neighbors4(px, py, n=self.n)):
                breeze = True
                break

        if (self.ax, self.ay) == self.gold and (not self.has_gold):
            glitter = True

        return {
            "Stench": stench,
            "Breeze": breeze,
            "Glitter": glitter,
            "Bump": self.last_bump,
            "Scream": self.last_scream
        }

    def _update_kb_from_percepts(self, p: Dict[str,bool]):
        cur = (self.ax, self.ay)
        self.kb[cur]["visited"] = True
        self.kb[cur]["no_pit"] = True
        self.kb[cur]["no_wumpus"] = True
        if p["Glitter"]:
            self.kb[cur]["known_gold"] = True

        adj = list(neighbors4(self.ax, self.ay, n=self.n))

        if p["Breeze"]:
            for nb in adj:
                if not self.kb[nb]["no_pit"]:
                    self.kb[nb]["poss_pit"] = True
        else:
            for nb in adj:
                self.kb[nb]["no_pit"] = True
                self.kb[nb]["poss_pit"] = False

        if self.wumpus_alive:
            if p["Stench"]:
                for nb in adj:
                    if not self.kb[nb]["no_wumpus"]:
                        self.kb[nb]["poss_wumpus"] = True
            else:
                for nb in adj:
                    self.kb[nb]["no_wumpus"] = True
                    self.kb[nb]["poss_wumpus"] = False
        else:
            for x in range(1, self.n+1):
                for y in range(1, self.n+1):
                    self.kb[(x,y)]["no_wumpus"] = True
                    self.kb[(x,y)]["poss_wumpus"] = False

    def _die(self, reason: str):
        self.dead = True
        self.done = True
        self.dead_at = (self.ax, self.ay)
        self.score -= 1000
        self.log.append(f"💀 {reason}  (-1000)")

    def act(self, action: str):
        if self.done:
            self.log.append("Episode already ended. Press Reset to start again.")
            return

        self.last_bump = False
        self.last_scream = False

        a = action.strip()
        if a not in ["Forward","TurnLeft","TurnRight","Grab","Shoot","Climb"]:
            self.log.append(f"Unknown action: {a}")
            return

        self.steps += 1
        self.score -= 1

        if a == "TurnLeft":
            i = DIRS.index(self.facing)
            self.facing = DIRS[(i - 1) % 4]
            self.log.append(f"↩️ TurnLeft -> facing {self.facing}")

        elif a == "TurnRight":
            i = DIRS.index(self.facing)
            self.facing = DIRS[(i + 1) % 4]
            self.log.append(f"↪️ TurnRight -> facing {self.facing}")

        elif a == "Forward":
            dx, dy = DIR_VEC[self.facing]
            nx, ny = self.ax + dx, self.ay + dy
            if not in_bounds(nx, ny, self.n):
                self.last_bump = True
                self.log.append("🧱 Forward -> Bump (hit wall), stayed in place.")
            else:
                self.ax, self.ay = nx, ny
                self.visited.add((self.ax, self.ay))
                self.kb[(self.ax, self.ay)]["visited"] = True
                self.log.append(f"➡️ Forward -> moved to ({self.ax},{self.ay})")

                if (self.ax, self.ay) in self.pits:
                    self.kb[(self.ax, self.ay)]["known_pit"] = True
                    self._die("Fell into a bottomless pit!")
                elif self.wumpus_alive and (self.ax, self.ay) == self.wumpus:
                    self.kb[(self.ax, self.ay)]["known_wumpus"] = True
                    self._die("Eaten by the Wumpus!")

        elif a == "Grab":
            if (self.ax, self.ay) == self.gold and (not self.has_gold):
                self.has_gold = True
                self.kb[(self.ax, self.ay)]["known_gold"] = True
                self.kb[(self.ax, self.ay)]["gold_taken"] = True
                self.log.append("✨ Grab -> picked up the gold!")
            else:
                self.log.append("🤷 Grab -> no gold here.")

        elif a == "Shoot":
            if self.arrow_left <= 0:
                self.log.append("🏹 Shoot -> no arrows left.")
            else:
                self.arrow_left -= 1
                self.score -= 10
                self.log.append("🏹 Shoot -> fired the arrow! (-10)")

                dx, dy = DIR_VEC[self.facing]
                x, y = self.ax, self.ay
                hit = False
                hit_cell = None
                while True:
                    x, y = x + dx, y + dy
                    if not in_bounds(x, y, self.n):
                        break
                    if self.wumpus_alive and (x, y) == self.wumpus:
                        self.wumpus_alive = False
                        self.last_scream = True
                        hit = True
                        hit_cell = (x, y)
                        break
                if hit:
                    self.kb[hit_cell]["known_wumpus"] = True
                    self.kb[hit_cell]["wumpus_dead_here"] = True
                    self.log.append(f"😱 Scream! Wumpus killed at {hit_cell}.")
                else:
                    self.log.append("… Arrow hit a wall (or flew until boundary) without hitting Wumpus.")

        elif a == "Climb":
            if (self.ax, self.ay) != (1,1):
                self.log.append("🧗 Climb -> can only climb out from (1,1).")
            else:
                if self.has_gold:
                    self.score += 1000
                    self.log.append("🏁 Climb -> escaped WITH the gold! (+1000)")
                else:
                    self.log.append("🏁 Climb -> escaped without gold.")
                self.done = True
                self.escaped = True

        p = self._percepts()
        self._update_kb_from_percepts(p)
        self.log.append(f"Percepts: {percept_list(p)}")

    # ---------- Rendering ----------
    def _cell_bg(self, x, y):
        if self.dead and self.dead_at == (x,y):
            return "background:#ffdddd;"
        if self.escaped and (x,y) == (1,1):
            return "background:#ddffdd;"
        return ""

    def _agent_icon(self):
        carry = "💰" if self.has_gold else ""
        return f"🤖{DIR_ARROW[self.facing]}{carry}"

    def _cell_full(self, x, y) -> str:
        items = []
        if (x,y) == (self.ax, self.ay):
            if self.dead and self.dead_at == (x,y):
                items.append("💀")
            items.append(self._agent_icon())
        if (x,y) == self.gold and (not self.has_gold):
            items.append("💰")
        if (x,y) == self.wumpus:
            items.append("👾" if self.wumpus_alive else "💀👾")
        if (x,y) in self.pits:
            items.append("🕳️")
        return " ".join(items) if items else "·"

    def _cell_agent(self, x, y) -> str:
        cell = (x,y)
        kb = self.kb[cell]
        items = []

        if cell == (self.ax, self.ay):
            if self.dead and self.dead_at == cell:
                items.append("💀")
            items.append(self._agent_icon())

        if kb["known_gold"]:
            items.append("" if kb["gold_taken"] else "💰")
        if kb["known_wumpus"]:
            items.append("💀👾" if kb["wumpus_dead_here"] else "👾")
        if kb["known_pit"]:
            items.append("🕳️")

        if kb["visited"] and cell != (self.ax, self.ay):
            items.append("·")

        if cell != (self.ax, self.ay) and not (kb["known_gold"] or kb["known_wumpus"] or kb["known_pit"]):
            visited = kb["visited"]
            poss_p = kb["poss_pit"] and (not kb["no_pit"])
            poss_w = kb["poss_wumpus"] and (not kb["no_wumpus"])
            safe_inferred = (not visited) and kb["no_pit"] and kb["no_wumpus"]

            if safe_inferred:
                items.append("✓")
            elif poss_p and poss_w:
                items.append("P?/W?")
            elif poss_p:
                items.append("P?")
            elif poss_w:
                items.append("W?")
            elif not items:
                items.append("□")

        if not items:
            items.append("□")
        return " ".join(items)

    def render_html(self, show_world: bool) -> str:
        p = self._percepts()
        percept_line = " | ".join([f"{k}: {'1' if v else '0'}" for k,v in p.items()])

        status_bits = [
            f"Score: <b>{self.score}</b> (steps={self.steps})",
            f"Arrow: <b>{self.arrow_left}</b>",
            f"Carry💰: <b>{'Yes' if self.has_gold else 'No'}</b>",
            f"Done: <b>{'Yes' if self.done else 'No'}</b>",
        ]
        if self.dead:
            status_bits.append("<span style='color:#b00020'><b>DEAD</b></span>")
        if self.escaped:
            status_bits.append("<span style='color:#1b5e20'><b>ESCAPED</b></span>")
        status_html = " &nbsp; | &nbsp; ".join(status_bits)

        def make_grid(cell_fn):
            rows = []
            for y in range(self.n, 0, -1):
                tds = []
                for x in range(1, self.n+1):
                    bg = self._cell_bg(x,y)
                    style = (
                        "width:110px;height:62px;text-align:center;border:1px solid #999;"
                        "font-size:15px;white-space:nowrap;"
                        f"{bg}"
                    )
                    tds.append(f"<td style='{style}'>{cell_fn(x,y)}</td>")
                rows.append("<tr>" + "".join(tds) + "</tr>")
            return "<table style='border-collapse:collapse;'>" + "".join(rows) + "</table>"

        agent_grid = make_grid(self._cell_agent)

        # Build grid container depending on show_world (no space when hidden)
        if show_world:
            world_grid = make_grid(self._cell_full)
            grid_css = "grid-template-columns: 1fr 1fr;"
            world_col = f"""
              <div>
                <div style="margin-bottom:6px;"><b>World</b></div>
                {world_grid}
                <div style="font-size:12px;color:#555;margin-top:6px;">
                  🤖(+💰), 💰, 👾/💀👾, 🕳️, 💀 (red), (1,1) green if escaped.
                </div>
              </div>
            """
        else:
            grid_css = "grid-template-columns: 1fr;"
            world_col = ""

        # newest first log
        tail = self.log[-22:][::-1]
        log_html = "<br>".join([f"{len(self.log)-i}. {line}" for i, line in enumerate(tail)])

        html = f"""
        <div style="font-family: ui-sans-serif, system-ui, -apple-system; line-height:1.25;">
          <h3 style="margin:0 0 6px 0;">Wumpus World (4×4) — Interactive Demo</h3>

          <div style="margin:4px 0 8px 0; padding:8px; border:1px solid #ddd; border-radius:10px;">
            {status_html}<br>
            <span style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas;">
              Percepts(bits) → {percept_line}
            </span>
          </div>

          <div style="
               display:grid;
               {grid_css}
               gap:18px;
               align-items:start;
               ">
            <div>
              <div style="margin-bottom:6px;"><b>Agent</b></div>
              {agent_grid}
              <div style="font-size:12px;color:#555;margin-top:6px;">
                □ unknown, ✓ safe, P?/W? possible, · visited,
                💰 found, 👾 found, 💀👾 dead, 🕳️ pit,
                💀 dead agent (red), (1,1) green if escaped.
              </div>
            </div>
            {world_col}
          </div>

          <div style="margin-top:12px; padding:10px; border:1px solid #ddd; border-radius:10px;">
            <div style="margin-bottom:6px;"><b>Log (newest first)</b></div>
            <div style="font-family: ui-monospace, SFMono-Regular, Menlo, Monaco, Consolas; font-size:12px;">
              {log_html}
            </div>
          </div>
        </div>
        """
        return html


# ---------- UI ----------
_demo = WumpusWorld()
_show_world = True
ui_out = W.Output()

def refresh():
    with ui_out:
        ui_out.clear_output(wait=True)
        display(button_panel)
        display(HTML(_demo.render_html(show_world=_show_world)))

def do_reset(_=None):
    _demo.reset()
    refresh()

def do_toggle(_=None):
    global _show_world
    _show_world = not _show_world
    refresh()

def do_action(action: str):
    def _handler(_btn=None):
        _demo.act(action)
        refresh()
    return _handler

btn_forward = W.Button(description="Forward", layout=W.Layout(width="110px"))
btn_left    = W.Button(description="TurnLeft", layout=W.Layout(width="110px"))
btn_right   = W.Button(description="TurnRight", layout=W.Layout(width="110px"))
btn_grab    = W.Button(description="Grab", layout=W.Layout(width="110px"))
btn_shoot   = W.Button(description="Shoot", layout=W.Layout(width="110px"))
btn_climb   = W.Button(description="Climb", layout=W.Layout(width="110px"))
btn_reset   = W.Button(description="Reset", layout=W.Layout(width="110px"))
btn_toggle  = W.Button(description="Toggle World", layout=W.Layout(width="150px"))

btn_forward.on_click(do_action("Forward"))
btn_left.on_click(do_action("TurnLeft"))
btn_right.on_click(do_action("TurnRight"))
btn_grab.on_click(do_action("Grab"))
btn_shoot.on_click(do_action("Shoot"))
btn_climb.on_click(do_action("Climb"))
btn_reset.on_click(do_reset)
btn_toggle.on_click(do_toggle)

button_panel = W.VBox([
    W.HBox([btn_forward, btn_left, btn_right, btn_grab]),
    W.HBox([btn_shoot, btn_climb, btn_reset, btn_toggle]),
])

_demo.reset()
refresh()
display(ui_out)


Output()